In [2]:
import h5py
from pathlib import Path

path = "/Users/carlasequero/Desktop/trabajo-fin-master/solution/data/06_training_data/training_03.h5"

with h5py.File(path, "r") as f:
    print(list(f.keys()))

['X', 'X_err', 'gaia_source_id', 'gaia_wavelength_nm', 'sdss_obj_id', 'sdss_wavelength_aa', 'y']


In [3]:
import numpy as np
import pandas as pd
from astroquery.gaia import Gaia
from astropy.table import vstack

with h5py.File(path, "r") as f:
    gaia_ID = f["gaia_source_id"][:]

print(gaia_ID.shape)

Please be advised that the system will experience intermittent service interruptions next Monday (06-07-2026), between 9:30 AM and 12:00 PM, due to scheduled hardware maintenance.
(55621,)


In [4]:
def query_gaia_hr_data(source_ids, batch_size=500):
    all_results = []

    for start in range(0, len(source_ids), batch_size):
        batch = source_ids[start:start + batch_size]

        ids_str = ",".join(str(int(x)) for x in batch)

        query = f"""
        SELECT
            source_id,
            ra,
            dec,
            phot_g_mean_mag,
            phot_bp_mean_mag,
            phot_rp_mean_mag,
            bp_rp,
            parallax,
            parallax_error,
            teff_gspphot,
            logg_gspphot,
            mh_gspphot
        FROM gaiadr3.gaia_source
        WHERE source_id IN ({ids_str})
        """

        print(f"Consultando lote {start // batch_size + 1} / {int(np.ceil(len(source_ids) / batch_size))}")

        job = Gaia.launch_job_async(query)
        result = job.get_results()

        all_results.append(result)

    return vstack(all_results)

In [5]:
gaia_hr_table = query_gaia_hr_data(gaia_ID, batch_size=200)

print(gaia_hr_table[:5])
print("Total recuperado:", len(gaia_hr_table))

Consultando lote 1 / 279
INFO: Query finished. [astroquery.utils.tap.core]
Consultando lote 2 / 279
INFO: Query finished. [astroquery.utils.tap.core]
Consultando lote 3 / 279
INFO: Query finished. [astroquery.utils.tap.core]
Consultando lote 4 / 279
INFO: Query finished. [astroquery.utils.tap.core]
Consultando lote 5 / 279
INFO: Query finished. [astroquery.utils.tap.core]
Consultando lote 6 / 279
INFO: Query finished. [astroquery.utils.tap.core]
Consultando lote 7 / 279
INFO: Query finished. [astroquery.utils.tap.core]
Consultando lote 8 / 279
INFO: Query finished. [astroquery.utils.tap.core]
Consultando lote 9 / 279
INFO: Query finished. [astroquery.utils.tap.core]
Consultando lote 10 / 279
INFO: Query finished. [astroquery.utils.tap.core]
Consultando lote 11 / 279
INFO: Query finished. [astroquery.utils.tap.core]
Consultando lote 12 / 279


KeyboardInterrupt: 

In [6]:
gaia_hr_df = gaia_hr_table.to_pandas()

bp_rp = gaia_hr_df["bp_rp"]
g_mag = gaia_hr_df["phot_g_mean_mag"]
parallax = gaia_hr_df["parallax"]

abs_g_mag = g_mag + 5 * np.log10(parallax) - 10

valid_hr = (
    np.isfinite(bp_rp)
    & np.isfinite(abs_g_mag)
    & np.isfinite(parallax)
    & (parallax > 0)
)

plt.figure(figsize=(7, 8))

plt.scatter(
    bp_rp[valid_hr],
    abs_g_mag[valid_hr],
    s=2,
    alpha=0.4
)

plt.gca().invert_yaxis()

plt.xlabel("BP - RP")
plt.ylabel("Magnitud absoluta G")
plt.title("Diagrama HR de las fuentes Gaia del conjunto de datos")
plt.show()

NameError: name 'gaia_hr_table' is not defined